In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/AmesHousing.csv")
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
df.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt',
      

In [4]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Lot Frontage       490
Alley             2732
Mas Vnr Type      1775
Mas Vnr Area        23
Bsmt Qual           80
Bsmt Cond           80
Bsmt Exposure       83
BsmtFin Type 1      80
BsmtFin SF 1         1
BsmtFin Type 2      81
BsmtFin SF 2         1
Bsmt Unf SF          1
Total Bsmt SF        1
Electrical           1
Bsmt Full Bath       2
Bsmt Half Bath       2
Fireplace Qu      1422
Garage Type        157
Garage Yr Blt      159
Garage Finish      159
Garage Cars          1
Garage Area          1
Garage Qual        159
Garage Cond        159
Pool QC           2917
Fence             2358
Misc Feature      2824
dtype: int64

In [5]:
null_col_df.dtypes

Lot Frontage      float64
Alley                 str
Mas Vnr Type          str
Mas Vnr Area      float64
Bsmt Qual             str
Bsmt Cond             str
Bsmt Exposure         str
BsmtFin Type 1        str
BsmtFin SF 1      float64
BsmtFin Type 2        str
BsmtFin SF 2      float64
Bsmt Unf SF       float64
Total Bsmt SF     float64
Electrical            str
Bsmt Full Bath    float64
Bsmt Half Bath    float64
Fireplace Qu          str
Garage Type           str
Garage Yr Blt     float64
Garage Finish         str
Garage Cars       float64
Garage Area       float64
Garage Qual           str
Garage Cond           str
Pool QC               str
Fence                 str
Misc Feature          str
dtype: object

In [6]:
df['MS SubClass'] = df['MS SubClass'].astype('str')

In [7]:
numeric_cols = df.select_dtypes(include='number').columns
categorical_cols = df.select_dtypes(exclude='number').columns
df[(df[numeric_cols] < 0).any(axis=1)] # make sure all numeric columns are nonnegative

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [8]:
values = {
    # "Lot Frontage": imputation,
    "Alley": "No Alley Access",
    "Mas Vnr Type": "None",
    # "Mas Vnr Area": imputation,
    "Bsmt Qual": "No Basement",
    "Bsmt Cond": "No Basement",
    "Bsmt Exposure" : "No Basement",
    "BsmtFin Type 1" : "No Basement",
    # "BsmtFin SF 1" : imputation,
    "BsmtFin Type 2" : "No Basement",
    # "BsmtFin SF 2" : imputation,
    # "Bsmt Unf SF" : imputation,
    # "Total Bsmt SF" : imputation,
    # "Electrical" : might actually be missing,
    # "Bsmt Full Bath" : imputation,
    # "Bsmt Half Bath" : imputation,
    "Fireplace Qu" : "No Fireplace",
    "Garage Type" : "No Garage",
    # "Garage Yr Blt" : might actually be missing,
    "Garage Finish" : "No Garage",
    # "Garage Cars" : imputation,
    # "Garage Area" : imputation,
    "Garage Qual" : "No Garage",
    "Garage Cond" : "No Garage",
    "Pool QC" : "No Pool",
    "Fence" : "No Fence",
    "Misc Feature" : "None"
}

In [9]:
inconsistency1 = (df['Mas Vnr Type'].isna()) & (df['Mas Vnr Area'] > 0) # if Type is NA AND Area > 0
inconsistency2 = (~df['Mas Vnr Type'].isna()) & (df['Mas Vnr Area'] == 0) # if Type is not NA AND Area == 0

In [10]:
df[df['Mas Vnr Area'] < 0] # area should be nonnegative

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [11]:
df[inconsistency1][['Mas Vnr Type', 'Mas Vnr Area']]

,Mas Vnr Type,Mas Vnr Area
363,NaN,344.0
403,NaN,312.0
441,NaN,285.0
1861,NaN,1.0
1913,NaN,1.0
2003,NaN,1.0
2528,NaN,288.0


In [12]:
df[inconsistency2][['Mas Vnr Type', 'Mas Vnr Area']]

,Mas Vnr Type,Mas Vnr Area
1640,BrkFace,0.0
1740,BrkFace,0.0
1785,Stone,0.0


In [13]:
df.loc[inconsistency1, 'Mas Vnr Type'] = 'Unknown'
df.loc[inconsistency2, 'Mas Vnr Type'] = 'None'

In [14]:
bsmt_cat_var = ['Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2']
bsmt_num_var = ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath']
bsmt_vars = bsmt_cat_var + bsmt_num_var
bsmt_row_nas = df[bsmt_cat_var].isna().sum(axis=1)
inconsistency3 = (bsmt_row_nas < len(bsmt_cat_var)) & (bsmt_row_nas > 0) # the bsmt categorical features contradict each other
inconsistency4 = (df[bsmt_cat_var].isna().sum(axis=1) == len(bsmt_cat_var)) & (df[bsmt_num_var].sum(axis=1) > 0) # bsmt numerical features contradict bsmt categorical features

In [15]:
df[df[bsmt_num_var].sum(axis=1) < 0] # all numeric features should be nonnegative

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [16]:
df[inconsistency3][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
66,Gd,TA,NaN,Unf,Unf,0.0,0.0,1595.0,1595.0,0.0,0.0
444,Gd,TA,No,GLQ,NaN,1124.0,479.0,1603.0,3206.0,1.0,0.0
1796,Gd,TA,NaN,Unf,Unf,0.0,0.0,725.0,725.0,0.0,0.0
2779,Gd,TA,NaN,Unf,Unf,0.0,0.0,936.0,936.0,0.0,0.0


In [17]:
df[inconsistency4][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath


In [18]:
df.loc[inconsistency3, bsmt_cat_var] = df.loc[inconsistency3, bsmt_cat_var].fillna('Unknown')

In [19]:
# Existence of Garage Consistency Checks
garage_cat_var = ['Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']
garage_num_var = ['Garage Yr Blt', 'Garage Cars', 'Garage Area']
garage_vars = garage_cat_var + garage_num_var
garage_row_nas = df[garage_cat_var].isna().sum(axis=1)

inconsistency5 = (df[garage_cat_var].isna().all(axis=1)) & (df[garage_num_var].sum(axis=1) > 0) # garage numerical features contradict garage categorical features
inconsistency6 = (garage_row_nas < len(garage_cat_var)) & (garage_row_nas > 0) # the garage categorical features contradict each other

In [20]:
# garage categorical features contradict each other doesn't rule out the possibility
# the structure actually exists, these values may actually be missing if the
# numerical features hint at the existence of the structure. If they do hint at the
# existence of the structure, then fill with 'Unknown'

In [21]:
df[inconsistency5][garage_vars + ['Year Built', 'Year Remod/Add']]

,Garage Type,Garage Finish,Garage Qual,Garage Cond,Garage Yr Blt,Garage Cars,Garage Area,Year Built,Year Remod/Add


In [22]:
df[inconsistency6][garage_vars + ['Year Built', 'Year Remod/Add']]

,Garage Type,Garage Finish,Garage Qual,Garage Cond,Garage Yr Blt,Garage Cars,Garage Area,Year Built,Year Remod/Add
1356,Detchd,NaN,NaN,NaN,NaN,1.0,360.0,1910,1983
2236,Detchd,NaN,NaN,NaN,NaN,NaN,NaN,1923,1999


In [23]:
garage_cat_noyr = [col for col in garage_cat_var if col != 'Garage Yr Blt']

In [24]:
df.loc[inconsistency6, garage_cat_noyr] = df.loc[inconsistency6, garage_cat_noyr].fillna('Unknown')

In [25]:
df[(df['Fireplaces'] > 0) & (df['Fireplace Qu'].isna())]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [26]:
df[(df['Pool Area'] > 0) & (df['Pool QC'].isna())]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [27]:
df[(df['Misc Val'] > 0) & (df['Misc Feature'].isna())]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [28]:
bsmt_total_check = df['BsmtFin SF 1'] + df['BsmtFin SF 2'] + df['Bsmt Unf SF'] == df['Total Bsmt SF']

In [29]:
df[bsmt_total_check][bsmt_vars].shape[0]

2929

In [30]:
df[~bsmt_total_check][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
1341,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
live_area_check = df['1st Flr SF'] + df['2nd Flr SF'] + df['Low Qual Fin SF'] == df['Gr Liv Area']

In [32]:
df[live_area_check].shape[0]

2930

In [33]:
garage_num_var

['Garage Yr Blt', 'Garage Cars', 'Garage Area']

In [34]:
# Garage Cars and Area Consistency Check
garage_consist_check1 = (df['Garage Cars'] > 0) & (df['Garage Area'] > 0)
garage_consist_check2 = (df['Garage Cars'] == 0) & (df['Garage Area'] == 0)
df[~garage_consist_check1][['Garage Cars', 'Garage Area']]

,Garage Cars,Garage Area
27,0.0,0.0
119,0.0,0.0
125,0.0,0.0
129,0.0,0.0
130,0.0,0.0
...,...,...
2913,0.0,0.0
2916,0.0,0.0
2918,0.0,0.0
2919,0.0,0.0


In [35]:
df[garage_consist_check2][['Garage Cars', 'Garage Area']]

,Garage Cars,Garage Area
27,0.0,0.0
119,0.0,0.0
125,0.0,0.0
129,0.0,0.0
130,0.0,0.0
...,...,...
2913,0.0,0.0
2916,0.0,0.0
2918,0.0,0.0
2919,0.0,0.0


In [36]:
df[~garage_consist_check1 & ~garage_consist_check2][['Garage Cars', 'Garage Area']]

,Garage Cars,Garage Area
2236,NaN,NaN


In [37]:
year_consist_check1 = (df['Year Remod/Add'] >= df['Year Built'])
index1 = df[~year_consist_check1].index
df[~year_consist_check1][['Year Remod/Add', 'Year Built', 'Yr Sold', 'Garage Yr Blt']]

,Year Remod/Add,Year Built,Yr Sold,Garage Yr Blt
850,2001,2002,2009,2002.0


In [38]:
year_consist_check2 = df['Yr Sold'] >= df['Year Built']
index2 = df[~year_consist_check2].index
df[~year_consist_check2][['Year Remod/Add', 'Year Built', 'Yr Sold', 'Garage Yr Blt']]

,Year Remod/Add,Year Built,Yr Sold,Garage Yr Blt
2180,2009,2008,2007,2008.0


In [39]:
year_consist_check3 = df['Yr Sold'] >= df['Garage Yr Blt']
index3 = df[~year_consist_check3 & ~df['Garage Yr Blt'].isna()].index
df[~year_consist_check3 & ~df['Garage Yr Blt'].isna()][['Year Remod/Add', 'Year Built', 'Yr Sold', 'Garage Yr Blt']]

,Year Remod/Add,Year Built,Yr Sold,Garage Yr Blt
2180,2009,2008,2007,2008.0
2260,2007,2006,2007,2207.0


In [40]:
df.loc[index1, 'Year Remod/Add'] = np.nan
df.loc[index2, 'Yr Sold'] = np.nan
df.loc[index3, 'Garage Yr Blt'] = np.nan

In [41]:
df.loc[list(index1) + list(index2) + list(index3), ['Year Remod/Add', 'Yr Sold', 'Year Built', 'Garage Yr Blt']]

,Year Remod/Add,Yr Sold,Year Built,Garage Yr Blt
850,NaN,2009.0,2002,2002.0
2180,2009.0,NaN,2008,NaN
2180,2009.0,NaN,2008,NaN
2260,2007.0,2007.0,2006,NaN


In [42]:
df.fillna(value=values, inplace=True)

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,No Alley Access,IR1,Lvl,...,0,No Pool,No Fence,None,0,5,2010.0,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,No Alley Access,Reg,Lvl,...,0,No Pool,MnPrv,None,0,6,2010.0,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,No Alley Access,IR1,Lvl,...,0,No Pool,No Fence,Gar2,12500,6,2010.0,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,No Alley Access,Reg,Lvl,...,0,No Pool,No Fence,None,0,4,2010.0,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,No Alley Access,IR1,Lvl,...,0,No Pool,MnPrv,None,0,3,2010.0,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,No Alley Access,IR1,Lvl,...,0,No Pool,GdPrv,None,0,3,2006.0,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,No Alley Access,IR1,Low,...,0,No Pool,MnPrv,None,0,6,2006.0,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,No Alley Access,Reg,Lvl,...,0,No Pool,MnPrv,Shed,700,7,2006.0,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,No Alley Access,Reg,Lvl,...,0,No Pool,No Fence,None,0,4,2006.0,WD,Normal,170000


In [43]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Lot Frontage      490
Year Remod/Add      1
Mas Vnr Area       23
BsmtFin SF 1        1
BsmtFin SF 2        1
Bsmt Unf SF         1
Total Bsmt SF       1
Electrical          1
Bsmt Full Bath      2
Bsmt Half Bath      2
Garage Yr Blt     161
Garage Cars         1
Garage Area         1
Yr Sold             1
dtype: int64

In [44]:
df[df['Total Bsmt SF'].isna()][bsmt_cat_var + bsmt_num_var]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
1341,No Basement,No Basement,No Basement,No Basement,No Basement,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
df[df['Bsmt Full Bath'].isna()][bsmt_cat_var + bsmt_num_var]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
1341,No Basement,No Basement,No Basement,No Basement,No Basement,NaN,NaN,NaN,NaN,NaN,NaN
1497,No Basement,No Basement,No Basement,No Basement,No Basement,0.0,0.0,0.0,0.0,NaN,NaN


In [46]:
df[df['Bsmt Half Bath'].isna()][bsmt_cat_var + bsmt_num_var]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
1341,No Basement,No Basement,No Basement,No Basement,No Basement,NaN,NaN,NaN,NaN,NaN,NaN
1497,No Basement,No Basement,No Basement,No Basement,No Basement,0.0,0.0,0.0,0.0,NaN,NaN


In [47]:
mask = df['Bsmt Qual'].eq('No Basement')
df.loc[mask, bsmt_num_var] = 0

In [48]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Lot Frontage      490
Year Remod/Add      1
Mas Vnr Area       23
Electrical          1
Garage Yr Blt     161
Garage Cars         1
Garage Area         1
Yr Sold             1
dtype: int64

In [49]:
df.loc[df['Mas Vnr Area'].isna(), 'Mas Vnr Area'] = 0

### Imputation

In [50]:
# group-based imputation
df['Lot Frontage'] = df['Lot Frontage'].fillna(
    df.groupby(['Neighborhood', 'Lot Config'])['Lot Frontage'].transform('median')
)

# global imputation for the 7 rows still missing
df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

In [51]:
df.loc[df['Electrical'].isna(), 'Electrical'] = df['Electrical'].mode()[0]

In [52]:
garage_vars = ['Garage Type', 'Garage Yr Blt', 'Garage Finish','Garage Cars', 'Garage Area', 'Garage Qual', 'Garage Cond']
cond1 = df['Garage Yr Blt'].isna()
cond2 = df['Garage Type'] == 'No Garage'
# No garage, 0
df.loc[cond1 & cond2, 'Garage Yr Blt'] = 0

# Has garage but year missing, impute with Year Built
df.loc[cond1 & ~cond2, 'Garage Yr Blt'] = df.loc[cond1 & ~cond2, 'Year Built']

In [53]:
df.loc[df['Year Remod/Add'].isna(), 'Year Remod/Add'] = df.loc[df['Year Remod/Add'].isna(), 'Year Built']
df.loc[df['Yr Sold'].isna(), 'Yr Sold'] = df.loc[df['Yr Sold'].isna(), ['Year Built', 'Garage Yr Blt', 'Year Remod/Add']].max(axis=1)

In [54]:
df.groupby('Garage Type')['Garage Cars'].transform('median')

0       2.0
1       2.0
2       2.0
3       2.0
4       2.0
       ... 
2925    2.0
2926    2.0
2927    0.0
2928    2.0
2929    2.0
Name: Garage Cars, Length: 2930, dtype: float64

In [55]:
# If No Garage, 0
df.loc[(df['Garage Cars'].isna()) & (df['Garage Type'] == 'No Garage'), 'Garage Cars'] = 0
df.loc[(df['Garage Area'].isna()) & (df['Garage Type'] == 'No Garage'), 'Garage Area'] = 0

# If garage, impute with median grouped by Garage Type
df.loc[df['Garage Cars'].isna(), 'Garage Cars'] = df.groupby('Garage Type')['Garage Cars'].transform('median')
df.loc[df['Garage Area'].isna(), 'Garage Area'] = df.groupby('Garage Type')['Garage Area'].transform('median')

In [56]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Series([], dtype: float64)

In [57]:
df.to_csv("../data/ames_cleaned.csv", index=False)